# Cortex Analyst — Natural Language Sales Q&A

## Use Case Overview

**Cortex Analyst** lets users ask business questions in plain English and get back accurate SQL + results — without writing a single query. This is a high-value capability for Solutions Architects to demonstrate to data teams, executives, and non-technical stakeholders.

**Why it matters for a SA:**
- Eliminates the "I need a data analyst to pull this report" bottleneck
- Reduces dashboard proliferation by enabling ad-hoc exploration
- Powered by a *semantic model* you define — so answers are grounded in your business logic, not guesswork

**Dataset:** TPC-H SF1 from `SNOWFLAKE_SAMPLE_DATA` — a standard benchmarking dataset representing a global order-management system (~6M orders, ~24M line items).

**Architecture:**
```
User question (natural language)
    → Cortex Analyst API
        → Reads semantic model YAML (staged)
            → Generates SQL
                → Executes against GENAI_LEARNING.PUBLIC views
                    → Returns results + explanation
```

## Step 1 — Setup & Imports

Connect to Snowflake and verify the views created by `setup.sql` are accessible.
Ensure you have run `setup.sql` and uploaded `sales_semantic_model.yaml` to the `@semantic_models` stage.

In [ ]:
import os
import json
import requests
import pandas as pd
import snowflake.connector
from snowflake.connector import DictCursor

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Upload the Semantic Model

The semantic model is a YAML file that describes your tables, columns, relationships, and verified queries.
Cortex Analyst reads this file to understand your data before generating SQL.

Run the cell below to upload `sales_semantic_model.yaml` to the `@semantic_models` stage.

In [ ]:
cur = conn.cursor()
cur.execute("PUT file://sales_semantic_model.yaml @GENAI_LEARNING.PUBLIC.semantic_models OVERWRITE=TRUE AUTO_COMPRESS=FALSE")
for row in cur.fetchall():
    print(row)

## Step 3 — Data Exploration

Before asking questions, understand what data the views expose.

In [ ]:
orders_df = pd.read_sql("SELECT * FROM orders_vw LIMIT 5", conn)
print("Orders view shape:", pd.read_sql("SELECT COUNT(*) AS n FROM orders_vw", conn).iloc[0,0], "rows")
orders_df

## Step 4 — Ask Questions with Cortex Analyst

Cortex Analyst is invoked via the Snowflake REST API. The helper function below:
1. Builds the API request with your question and the path to the semantic model
2. Calls the `/api/v2/cortex/analyst/message` endpoint
3. Parses and displays the generated SQL and result

**Key parameters:**
- `semantic_model_file`: path to your YAML in `@stage/file.yaml` format
- `messages`: conversation history (supports multi-turn)

In [ ]:
def ask_cortex_analyst(question: str, conversation_history: list = None) -> dict:
    token = conn.rest.token
    host = conn.host

    messages = conversation_history or []
    messages.append({"role": "user", "content": [{"type": "text", "text": question}]})

    payload = {
        "messages": messages,
        "semantic_model_file": "@GENAI_LEARNING.PUBLIC.semantic_models/sales_semantic_model.yaml"
    }

    response = requests.post(
        f"https://{host}/api/v2/cortex/analyst/message",
        json=payload,
        headers={
            "Authorization": f"Snowflake Token=\"{token}\"",
            "Content-Type": "application/json",
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def display_analyst_response(response: dict):
    for item in response.get("message", {}).get("content", []):
        if item["type"] == "text":
            print("Analyst:", item["text"])
        elif item["type"] == "sql":
            print("\nGenerated SQL:\n", item["statement"])
            df = pd.read_sql(item["statement"], conn)
            display(df)
    return response.get("message", {}).get("content", [])

### Question 1 — Simple aggregation
Ask a basic revenue question to verify the model is working.

In [ ]:
resp = ask_cortex_analyst("What is the total revenue by month for 1995?")
history = display_analyst_response(resp)

### Question 2 — Join across tables
Ask something that requires joining orders to customers.

In [ ]:
resp2 = ask_cortex_analyst("Which market segment generates the most revenue?")
display_analyst_response(resp2)

### Question 3 — Multi-turn follow-up
Cortex Analyst supports multi-turn conversation — follow up on the previous answer.

In [ ]:
resp3 = ask_cortex_analyst("Now break that down by year", conversation_history=history)
display_analyst_response(resp3)

## Step 5 — Interpretation & SA Tips

**What just happened:**
- Cortex Analyst parsed your question, matched it against the semantic model's measures/dimensions, and generated valid SQL
- The semantic model defines *business logic* (e.g. `net_revenue = extended_price * (1 - discount)`) so the LLM doesn't have to guess
- `verified_queries` in the YAML act as few-shot examples that significantly improve accuracy

**Production tips for SAs:**
1. **Start with verified queries** — they anchor the model to your known-good SQL patterns
2. **Be explicit with descriptions** — the richer the column descriptions, the better the SQL quality
3. **Use `relationships`** — without explicit join definitions, the model cannot join tables reliably
4. **Test with edge cases** — questions about time ranges, top-N, and filters are the most common failure points
5. **Semantic views vs stage files** — for production, consider `CREATE SEMANTIC VIEW` (no stage needed) instead of a YAML file on a stage